# Task 6: Sustainability-Inclusive Evaluation

Instrument Task 3's LoRA fine-tuning run with energy/carbon tracking (`codecarbon`), and report compute-cost vs. accuracy tradeoffs.

- **Run under instrumentation:** Task 3's exact recipe (base CLIP `ViT-B-32`, LoRA fine-tune on 200 RSICD images, 1 epoch).
- **What's tracked:** three separate stages — baseline (untuned) evaluation, LoRA training, and post-fine-tune evaluation — each wrapped in its own `codecarbon` `EmissionsTracker`, so training cost and inference cost aren't lumped together.
- **Accuracy proxy:** the same purity-aware retrieval harness from Task 3/4 (mean purity over the top-3 for 6 held-out queries).
- **Caveat:** this runs on a shared/virtualized Colab GPU, so `codecarbon`'s hardware-based energy estimates are approximate, not lab-grade — useful as a relative baseline (training vs. inference, cost vs. purity gained), not an absolute number to publish.
- **Flow:** load dataset → load base CLIP → evaluation harness → track baseline eval → apply LoRA → track training → track post-fine-tune eval → compute-cost vs. accuracy summary → findings write-up (bottom of notebook).

### Step 1: Setup

In [ ]:
%pip install open_clip_torch peft codecarbon
# Colab preinstalls an old torchao (0.10.0); peft>=0.15 requires torchao>=0.16.0 at import
# time even though this notebook never uses quantization (same fix as Task 3/4).
%pip install -U torchao

In [ ]:
import re
import random
from collections import defaultdict

import torch
import torch.nn.functional as F
import open_clip
import pandas as pd

from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from codecarbon import EmissionsTracker

### Step 2: Load RSICD and define the fine-tuning subset + held-out evaluation sample

Same seeded split as Task 3/4, so the accuracy side of this experiment is directly comparable to those runs.

In [ ]:
ds = load_dataset("arampacha/rsicd")
ds

In [ ]:
random.seed(0)

NUM_FT_SAMPLES = 200
ft_indices = random.sample(range(len(ds["train"])), NUM_FT_SAMPLES)
ft_indices_set = set(ft_indices)

CATEGORY_RE = re.compile(r'^([a-zA-Z]+)_\d+\.jpg$')

def category_from_filename(filename):
    name = filename.split('/')[-1]
    match = CATEGORY_RE.match(name)
    return match.group(1) if match else None

by_category = defaultdict(list)
uncategorized = 0
for idx, fname in enumerate(ds['train']['filename']):
    if idx in ft_indices_set:
        continue
    cat = category_from_filename(fname)
    if cat is None:
        uncategorized += 1
    else:
        by_category[cat].append(idx)

eval_sample_indices = []
for cat, idxs in by_category.items():
    eval_sample_indices.extend(random.sample(idxs, min(5, len(idxs))))

print(f"Fine-tuning subset: {len(ft_indices)} images")
print(f"Held-out eval sample: {len(eval_sample_indices)} images across {len(by_category)} labeled categories")

### Step 3: Load the base (generic) CLIP model

In [ ]:
model_name = "ViT-B-32"

base_model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained="laion2b_s34b_b79k")
tokenizer = open_clip.get_tokenizer(model_name)
base_model = base_model.cuda().eval()

### Step 4: Evaluation harness (hit-rate + purity, same as Task 3/4)

In [ ]:
queries_with_expected = [
    ("a large bridge crossing a river", ["bridge"]),
    ("an airport with parked airplanes", ["airport"]),
    ("a dense green forest", ["forest"]),
    ("farmland divided into rectangular plots", ["farmland"]),
    ("a stadium with a running track", ["stadium"]),
    ("a residential area with many houses", ["denseresidential", "mediumresidential", "sparseresidential"]),
]
queries = [q for q, _ in queries_with_expected]

def evaluate_retrieval(model, label, verbose=True):
    model.eval()
    sample_images = [ds['train'][i]['image'] for i in eval_sample_indices]
    sample_filenames = [ds['train'][i]['filename'] for i in eval_sample_indices]
    sample_categories = [category_from_filename(f) for f in sample_filenames]

    image_tensors = torch.stack([preprocess(img) for img in sample_images]).to("cuda")

    with torch.no_grad(), torch.autocast("cuda"):
        bank_image_features = model.encode_image(image_tensors)
        bank_image_features /= bank_image_features.norm(dim=-1, keepdim=True)

        query_tokens = tokenizer(queries).to("cuda")
        query_features = model.encode_text(query_tokens)
        query_features /= query_features.norm(dim=-1, keepdim=True)

    sims = query_features @ bank_image_features.T

    if verbose:
        print(f"\n{'=' * 20} {label} {'=' * 20}")

    hits = 0
    purities = []
    for qi, (q, expected_cats) in enumerate(queries_with_expected):
        top_idx = sims[qi].topk(3).indices.tolist()
        top_cats = [sample_categories[ii] for ii in top_idx]
        hit = any(c in expected_cats for c in top_cats)
        hits += hit
        purity = sum(1 for c in top_cats if c in expected_cats) / len(top_cats)
        purities.append(purity)
        if verbose:
            mark = "\u2713" if hit else "\u2717"
            print(f"  {mark} {q!r} -> top3 categories: {top_cats}  (purity={purity:.2f})")

    mean_purity = sum(purities) / len(purities)
    if verbose:
        print(f"  Score: {hits}/{len(queries)}  |  Mean purity: {mean_purity:.3f}")

    return hits, mean_purity

### Step 5: Emissions-tracking helper

Wraps a zero-argument callable in its own `codecarbon` tracker (context-manager style: `start()` on entry, `stop()` on exit), and records duration/energy/CO2 alongside whatever the callable returns.

In [ ]:
stage_records = []

def track_stage(stage_name, fn):
    with EmissionsTracker(project_name=f"task6_{stage_name}", log_level="error", save_to_file=False) as tracker:
        result = fn()
    data = tracker.final_emissions_data
    stage_records.append({
        "stage": stage_name,
        "duration_s": data.duration,
        "energy_kwh": data.energy_consumed,
        "co2_kg": data.emissions,
    })
    print(f"[{stage_name}] duration={data.duration:.1f}s  energy={data.energy_consumed * 1000:.4f} Wh  CO2={data.emissions * 1000:.4f} g")
    return result

### Step 6: Stage 1 — track the baseline (untuned) evaluation

In [ ]:
base_score, base_purity = track_stage("baseline_eval", lambda: evaluate_retrieval(base_model, "Base CLIP (untuned)"))

### Step 7: Apply LoRA (setup only — negligible compute, not tracked)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["out_proj", "c_fc", "c_proj"],
    lora_dropout=0.05,
    bias="none",
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()
clip = lora_model.base_model.model

### Step 8: Stage 2 — track LoRA fine-tuning (1 epoch, 200 samples — Task 3's recipe)

In [ ]:
BATCH_SIZE = 16
LR = 1e-4

def run_training():
    optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=LR)

    shuffled_ft_indices = ft_indices.copy()
    random.shuffle(shuffled_ft_indices)

    clip.train()
    losses = []
    num_batches = (len(shuffled_ft_indices) + BATCH_SIZE - 1) // BATCH_SIZE

    for b, start in enumerate(range(0, len(shuffled_ft_indices), BATCH_SIZE), 1):
        batch_idx = shuffled_ft_indices[start:start + BATCH_SIZE]
        batch_images = torch.stack([preprocess(ds['train'][i]['image']) for i in batch_idx]).to("cuda")
        batch_captions = [ds['train'][i]['captions'][0] for i in batch_idx]
        batch_text = tokenizer(batch_captions).to("cuda")

        with torch.autocast("cuda"):
            image_features = clip.encode_image(batch_images)
            text_features = clip.encode_text(batch_text)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

            logit_scale = clip.logit_scale.exp()
            logits_per_image = logit_scale * image_features @ text_features.T
            logits_per_text = logits_per_image.T

            labels = torch.arange(len(batch_idx), device="cuda")
            loss = (F.cross_entropy(logits_per_image, labels) + F.cross_entropy(logits_per_text, labels)) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        print(f"batch {b}/{num_batches}  loss={loss.item():.4f}")

    print(f"\nMean loss over the epoch: {sum(losses) / len(losses):.4f}")

track_stage("lora_training", run_training)

### Step 9: Stage 3 — track the post-fine-tune evaluation

In [ ]:
lora_score, lora_purity = track_stage("post_finetune_eval", lambda: evaluate_retrieval(clip, "LoRA fine-tuned CLIP"))

### Step 10: Compute-cost vs. accuracy summary

In [ ]:
purity_by_stage = {"baseline_eval": base_purity, "lora_training": None, "post_finetune_eval": lora_purity}
for r in stage_records:
    r["purity"] = purity_by_stage[r["stage"]]

stage_df = pd.DataFrame(stage_records)
print(stage_df.to_string(index=False))

purity_delta = lora_purity - base_purity
train_row = stage_df[stage_df.stage == "lora_training"].iloc[0]
eval_rows = stage_df[stage_df.stage != "lora_training"]
mean_eval_energy_kwh = eval_rows["energy_kwh"].mean()
mean_eval_duration_s = eval_rows["duration_s"].mean()

print(f"\nPurity improvement from fine-tuning: {purity_delta:+.3f} (0.0-1.0 scale)")
print(f"Training cost: {train_row.duration_s:.1f}s, {train_row.energy_kwh * 1000:.4f} Wh, {train_row.co2_kg * 1000:.4f} g CO2eq")
print(f"Mean single eval pass cost: {mean_eval_duration_s:.1f}s, {mean_eval_energy_kwh * 1000:.4f} Wh")
print(f"Training was ~{train_row.duration_s / mean_eval_duration_s:.1f}x longer and ~{train_row.energy_kwh / mean_eval_energy_kwh:.1f}x more energy-intensive than a single eval pass")
if purity_delta > 0:
    print(f"Energy cost per full purity point gained (illustrative, from this one run): {train_row.energy_kwh * 1000 / purity_delta:.2f} Wh")

## Step 11: Findings write-up

_TBD — to be filled in after running all cells and reviewing the actual output._